# KLUE-RoBERTa 70감정 분류 v3 — 풀옵션
- EDA 데이터 증강 (기존 데이터 3배)
- HuggingFace 공개 한국어 감정 데이터 추가
- 하이퍼파라미터 최적화
- 런타임 → GPU(T4) 설정 후 실행

In [1]:
!pip install -q transformers accelerate datasets scikit-learn

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/감정분석 모델/감정분석_70감정_추가증강.csv'
SAVE_PATH = '/content/drive/MyDrive/감정분석 모델/klue-roberta-70emotion-v3'

Mounted at /content/drive


In [3]:
import os, random, re
import numpy as np
import pandas as pd
import torch
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
    DataCollatorWithPadding
)

os.environ['WANDB_DISABLED'] = 'true'
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 — 런타임 GPU로 변경!"}')

GPU: Tesla T4


## 1단계: 기존 데이터 로딩

In [4]:
TEXT_COL = '문장 예시'
LABEL_COL = '감정명'

df = pd.read_csv(DATA_PATH, encoding='utf-8-sig').dropna(subset=[TEXT_COL, LABEL_COL])
df = df.rename(columns={TEXT_COL: 'text', LABEL_COL: 'label'})
df['text'] = df['text'].str.strip()
df['label'] = df['label'].str.strip()

# 충돌 텍스트 다수결 처리
df['_norm'] = df['text'].str.lower()
conflict_mask = df.groupby('_norm')['label'].transform('nunique') > 1
if conflict_mask.any():
    majority = df[conflict_mask].groupby('_norm')['label'].agg(lambda s: s.value_counts().index[0])
    df.loc[conflict_mask, 'label'] = df.loc[conflict_mask, '_norm'].map(majority)
df = df.drop_duplicates(['_norm', 'label'])[['text', 'label']].reset_index(drop=True)

print(f'기존 데이터: {len(df)}행, {df["label"].nunique()}클래스')
print(f'클래스당 평균: {df["label"].value_counts().mean():.0f}개')

기존 데이터: 26835행, 70클래스
클래스당 평균: 383개


## 2단계: HuggingFace 공개 한국어 감정 데이터 추가

In [5]:
from datasets import load_dataset

extra_frames = []

# ── Dataset 1: 감정 대화 데이터 (smilestyle) ────────────────────────────────
# 기쁨/슬픔/분노/공포/혐오/놀람 6개 감정, 한국어 일상 대화
try:
    ds1 = load_dataset('heegyu/korsts', split='train')  # fallback용
    print('korsts 로딩 실패 — 건너뜀')
except:
    pass

try:
    ds1 = load_dataset('smilegate-ai/kor_unsmile', split='train')
    print('kor_unsmile 로딩 실패 — 건너뜀')
except:
    pass

# ── Dataset 2: 감정 분류 데이터셋 ───────────────────────────────────────────
try:
    ds2 = load_dataset('hun3359/klue-bert-base-sentiment', split='train')
    # label: 0=부정, 1=중립, 2=긍정 — 우리 70감정과 직접 매핑 불가지만
    # 감정 표현 문장 자체는 활용 가능
    print(f'klue-sentiment 로딩: {len(ds2)}행')
except Exception as e:
    print(f'klue-sentiment 실패: {e}')

# ── Dataset 3: AI Hub 스타일 감정 데이터 ────────────────────────────────────
try:
    ds3 = load_dataset('jeanlee/kmhas_korean_hate_speech', split='train')
    print(f'kmhas 로딩: {len(ds3)}행')
except Exception as e:
    print(f'kmhas 실패: {e}')

# ── Dataset 4: 감정 레이블이 있는 한국어 대화 ────────────────────────────────
try:
    ds4 = load_dataset('psyche/kor_emotional_dialogue')
    tmp = pd.DataFrame(ds4['train'])
    print(f'kor_emotional_dialogue 로딩: {len(tmp)}행')
    print(f'컬럼: {tmp.columns.tolist()}')
    print(f'감정 종류: {tmp.iloc[:,1].unique()[:10]}')
except Exception as e:
    print(f'kor_emotional_dialogue 실패: {e}')

print('\n외부 데이터 탐색 완료')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


dataset_infos.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

data/valid-00000-of-00001.parquet:   0%|          | 0.00/290k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15005 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/3737 [00:00<?, ? examples/s]

kor_unsmile 로딩 실패 — 건너뜀
klue-sentiment 실패: Dataset 'hun3359/klue-bert-base-sentiment' doesn't exist on the Hub or cannot be accessed.


README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

kmhas_korean_hate_speech.py:   0%|          | 0.00/4.73k [00:00<?, ?B/s]

kmhas 실패: Dataset scripts are no longer supported, but found kmhas_korean_hate_speech.py
kor_emotional_dialogue 실패: Dataset 'psyche/kor_emotional_dialogue' doesn't exist on the Hub or cannot be accessed.

외부 데이터 탐색 완료


In [6]:
# ── 로딩된 외부 데이터를 우리 70감정 레이블에 매핑 ──────────────────────────
# kor_emotional_dialogue 기준 (감정 종류 확인 후 매핑)

OUR_LABELS = set(df['label'].unique())

# 외부 감정명 → 우리 감정명 매핑 테이블 (겹치는 것만)
EMOTION_MAP = {
    # 외부 레이블 : 우리 레이블 (우리 레이블에 없으면 None)
    '기쁨': '기쁨', '행복': '행복감', '슬픔': '슬픔', '분노': '분노',
    '공포': '두려움', '놀람': '놀라움', '혐오': '혐오감', '불안': '불안감',
    '외로움': '외로움', '우울': '우울감', '당황': '당황', '실망': '실망',
    '죄책감': '죄책감', '부끄러움': '부끄러움', '질투': '질투심',
    '감사': '감사함', '사랑': '사랑', '희망': '희망', '설렘': '설렘',
}
# 우리 레이블에 있는 것만 필터
EMOTION_MAP = {k: v for k, v in EMOTION_MAP.items() if v in OUR_LABELS}
print('매핑 가능한 외부 감정:', list(EMOTION_MAP.keys()))

extra_dfs = [df]  # 기존 데이터 먼저

try:
    tmp = pd.DataFrame(ds4['train'])
    # 컬럼명 확인 후 적절히 조정
    text_col = tmp.columns[0]
    label_col = tmp.columns[1]
    tmp = tmp.rename(columns={text_col: 'text', label_col: 'label'})
    tmp['label'] = tmp['label'].map(EMOTION_MAP)
    tmp = tmp.dropna(subset=['label'])[['text', 'label']]
    tmp['text'] = tmp['text'].astype(str).str.strip()
    print(f'외부 데이터 추가 가능: {len(tmp)}행')
    extra_dfs.append(tmp)
except Exception as e:
    print(f'외부 데이터 매핑 실패: {e}')

df_all = pd.concat(extra_dfs, ignore_index=True).drop_duplicates('text').reset_index(drop=True)
print(f'\n통합 후: {len(df_all)}행')

매핑 가능한 외부 감정: ['기쁨', '분노', '공포', '외로움', '우울', '당황', '실망', '설렘']
외부 데이터 매핑 실패: name 'ds4' is not defined

통합 후: 26835행


## 3단계: EDA (Easy Data Augmentation) 증강

In [7]:
def eda_augment(text: str, n: int = 2) -> list[str]:
    """단어 삭제/교환/반복으로 새 문장 생성"""
    results = []
    words = text.split()
    if len(words) < 3:
        return [text] * n

    for _ in range(n):
        op = random.choice(['delete', 'swap', 'repeat'])
        w = words.copy()

        if op == 'delete' and len(w) > 3:
            # 랜덤 단어 1개 삭제 (단 마지막 단어 제외 — 종결어미 보존)
            idx = random.randint(0, len(w) - 2)
            w.pop(idx)

        elif op == 'swap' and len(w) > 2:
            # 인접 단어 교환
            idx = random.randint(0, len(w) - 2)
            w[idx], w[idx + 1] = w[idx + 1], w[idx]

        elif op == 'repeat':
            # 감정 강조 표현 앞에 '정말 / 너무 / 진짜' 추가
            prefix = random.choice(['정말 ', '너무 ', '진짜 ', '완전 '])
            w = [prefix + w[0]] + w[1:]

        results.append(' '.join(w))
    return results


# 클래스당 목표: 최소 800개 (부족한 클래스만 증강)
TARGET_PER_CLASS = 800
augmented_rows = []

for label, group in df_all.groupby('label'):
    current = len(group)
    if current >= TARGET_PER_CLASS:
        continue
    needed = TARGET_PER_CLASS - current
    texts = group['text'].tolist()
    new_texts = []
    while len(new_texts) < needed:
        src = random.choice(texts)
        new_texts.extend(eda_augment(src, n=2))
    new_texts = new_texts[:needed]
    augmented_rows.append(pd.DataFrame({'text': new_texts, 'label': label}))

if augmented_rows:
    df_aug = pd.concat([df_all] + augmented_rows, ignore_index=True)
else:
    df_aug = df_all.copy()

df_aug = df_aug.drop_duplicates('text').sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f'증강 후: {len(df_aug)}행')
print(f'클래스당 평균: {df_aug["label"].value_counts().mean():.0f}, '
      f'최소: {df_aug["label"].value_counts().min()}')

증강 후: 52442행
클래스당 평균: 749, 최소: 691


## 4단계: 학습/검증 분할 & 토크나이징

In [8]:
encoder = LabelEncoder()
labels_encoded = encoder.fit_transform(df_aug['label'].tolist()).tolist()
texts_all = df_aug['text'].tolist()

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts_all, labels_encoded, test_size=0.15, random_state=SEED, stratify=labels_encoded
)
print(f'학습: {len(train_texts)}, 검증: {len(val_texts)}')

MODEL_NAME = 'klue/roberta-large'
MAX_LEN = 128
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class EmotionDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(texts, truncation=True, padding='max_length',
                             max_length=MAX_LEN, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {'input_ids': self.enc['input_ids'][idx],
                'attention_mask': self.enc['attention_mask'][idx],
                'labels': self.labels[idx]}

train_dataset = EmotionDataset(train_texts, train_labels)
val_dataset   = EmotionDataset(val_texts,   val_labels)
print('Dataset 생성 완료')

학습: 44575, 검증: 7867


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/752k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Dataset 생성 완료


## 5단계: 모델 & 학습

In [9]:
id2label = {i: l for i, l in enumerate(encoder.classes_)}
label2id = {l: i for i, l in enumerate(encoder.classes_)}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(encoder.classes_),
    id2label=id2label, label2id=label2id,
)
print(f'파라미터: {sum(p.numel() for p in model.parameters()):,}')

model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


파라미터: 336,728,134


In [10]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = np.argmax(pred.predictions, axis=1)
    return {'accuracy': float(accuracy_score(labels, preds)),
            'f1':       float(f1_score(labels, preds, average='weighted'))}

training_args = TrainingArguments(
    output_dir='/content/roberta_checkpoints',
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,          # v2보다 낮춤
    weight_decay=0.01,
    warmup_steps=500,
    lr_scheduler_type='cosine',
    label_smoothing_factor=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=100,
    fp16=True,
    seed=SEED,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)

In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,2.297674,2.163955,0.592221,0.620417
2,1.961567,1.983332,0.648913,0.669483
3,1.749491,1.849470,0.690733,0.701485
4,1.595302,1.777275,0.715775,0.722116
5,1.459974,1.717118,0.741452,0.751378
6,1.336527,1.660666,0.763696,0.764950
7,1.261363,1.631172,0.776026,0.776816
8,1.134471,1.615959,0.790899,0.792224
9,0.986053,1.599272,0.799542,0.799613
10,0.933654,1.592105,0.803737,0.803506


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=41790, training_loss=1.283402982275005, metrics={'train_runtime': 16066.7647, 'train_samples_per_second': 41.615, 'train_steps_per_second': 2.601, 'total_flos': 1.55813943180096e+17, 'train_loss': 1.283402982275005, 'epoch': 15.0})

In [12]:
results = trainer.evaluate()
print(f"\n최종 정확도: {results['eval_accuracy']:.4f}")
print(f"최종 F1:     {results['eval_f1']:.4f}")

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.792998,1.576060,15,0.813016,0.812776



최종 정확도: 0.8130
최종 F1:     0.8128


In [13]:
import json
os.makedirs(SAVE_PATH, exist_ok=True)
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

with open(f'{SAVE_PATH}/label_classes.json', 'w', encoding='utf-8') as f:
    json.dump(encoder.classes_.tolist(), f, ensure_ascii=False, indent=2)
with open(f'{SAVE_PATH}/metrics.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f'저장 완료: {SAVE_PATH}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

저장 완료: /content/drive/MyDrive/감정분석 모델/klue-roberta-70emotion-v3


## 6단계: 예측 테스트

In [14]:
from transformers import pipeline

pipe = pipeline('text-classification', model=model, tokenizer=tokenizer,
                device=0 if torch.cuda.is_available() else -1, top_k=3)

tests = [
    '오늘 너무 행복해서 눈물이 날 것 같아',
    '그 말 듣고 진짜 화가 많이 났어',
    '혼자 있으니까 외롭고 쓸쓸하다',
    '내일 발표가 있어서 너무 긴장돼',
    '갑자기 실수해서 너무 창피했어',
]
for sent in tests:
    top = pipe(sent)[0][0]
    print(f'{sent}\n  → {top["label"]} ({top["score"]:.3f})\n')

오늘 너무 행복해서 눈물이 날 것 같아
  → 기쁨 (0.515)

그 말 듣고 진짜 화가 많이 났어
  → 짜증 (0.421)

혼자 있으니까 외롭고 쓸쓸하다
  → 외로움 (0.764)

내일 발표가 있어서 너무 긴장돼
  → 스트레스 (0.927)

갑자기 실수해서 너무 창피했어
  → 자책 (0.207)

